# 90+ Gündem — MoneyPrinterTurbo Colab
Kalıcı kurulum: repo güncellenir, Pexels 404/403/timeout hataları tek klip yüzünden üretimi durdurmaz.

In [ ]:
import os, subprocess, re
from pathlib import Path

REPO_DIR = Path('/content/MoneyPrinterTurbo')
REPO_URL = 'https://github.com/harry0703/MoneyPrinterTurbo.git'

if (REPO_DIR / '.git').is_dir():
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard','HEAD'], check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'pull','--ff-only'], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f'{REPO_DIR} exists but is not a Git repository')
else:
    subprocess.run(['git','clone','--depth','1',REPO_URL,str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(['python','-m','pip','install','-q','uv','pyngrok'], check=True)
subprocess.run(['uv','python','install','3.11'], check=True)
subprocess.run(['uv','sync','--frozen','--python','3.11'], check=True)

# 90+ Gündem resilience patch. Applied after every upstream pull, so it survives Colab resets.
material_py = REPO_DIR / 'app/services/material.py'
text = material_py.read_text(encoding='utf-8')
start = text.find('def save_video(')
end = text.find('\ndef _search_videos_with_cache(', start)
if start < 0 or end < 0:
    raise RuntimeError('MoneyPrinterTurbo save_video function layout changed; patch safely aborted')
patched = '''def save_video(video_url: str, save_dir: str = '') -> str:
    if not save_dir:
        save_dir = utils.storage_dir('cache_videos')
    os.makedirs(save_dir, exist_ok=True)
    url_without_query = video_url.split('?', 1)[0]
    extension = os.path.splitext(url_without_query)[1] or '.mp4'
    video_path = os.path.join(save_dir, f"vid-{utils.md5(video_url)}{extension}")
    if os.path.exists(video_path) and os.path.getsize(video_path) > 0:
        try:
            clip = VideoFileClip(video_path)
            ok = clip.duration > 0 and clip.fps > 0
            clip.close()
            if ok:
                return video_path
        except Exception:
            pass
        try:
            os.remove(video_path)
        except OSError:
            pass
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/115 Safari/537.36"}
    for attempt in range(1, 4):
        try:
            with requests.get(video_url, headers=headers, proxies=config.proxy, verify=_get_tls_verify(), timeout=(30, 120), stream=True) as response:
                if response.status_code in (403, 404, 410):
                    logger.warning(f"stock clip unavailable HTTP {response.status_code}; skip candidate")
                    return ''
                response.raise_for_status()
                content_type = str(response.headers.get('content-type', '')).lower()
                if 'text/html' in content_type or 'application/json' in content_type:
                    logger.warning(f"stock clip returned non-video content ({content_type}); skip candidate")
                    return ''
                tmp_path = video_path + '.part'
                with open(tmp_path, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
                os.replace(tmp_path, video_path)
            clip = VideoFileClip(video_path)
            ok = clip.duration > 0 and clip.fps > 0
            clip.close()
            if ok:
                return video_path
            raise ValueError('downloaded stock clip is not a valid video')
        except (requests.Timeout, requests.ConnectionError) as exc:
            logger.warning(f"stock clip network error attempt {attempt}/3: {type(exc).__name__}")
            if attempt < 3:
                time.sleep(attempt * 2)
        except requests.HTTPError as exc:
            logger.warning(f"stock clip HTTP error; skip candidate: {exc.response.status_code if exc.response is not None else 'unknown'}")
            break
        except Exception as exc:
            logger.warning(f"invalid stock clip; skip candidate: {type(exc).__name__}")
            break
        finally:
            part = video_path + '.part'
            if os.path.exists(part):
                try: os.remove(part)
                except OSError: pass
    if os.path.exists(video_path):
        try: os.remove(video_path)
        except OSError: pass
    return ''
'''
material_py.write_text(text[:start] + patched + text[end:], encoding='utf-8')
subprocess.run(['uv','run','python','-m','py_compile',str(material_py)], check=True)
print('READY: MoneyPrinterTurbo + resilient stock-video fallback patch')


## Ngrok bağlantısı
Token yalnızca çalışma oturumunda tutulur.

In [ ]:
from getpass import getpass
from pyngrok import ngrok
ngrok.kill()
ngrok_token = getpass('Enter your ngrok authentication token: ').strip()
if not ngrok_token: raise ValueError('An ngrok authentication token is required')
ngrok.set_auth_token(ngrok_token)
del ngrok_token
print('ngrok authentication configured')


## WebUI başlat

In [ ]:
import time
from urllib.error import URLError
from urllib.request import urlopen
PORT=8501
LOG_PATH=Path('/content/moneyprinterturbo-webui.log')
p=globals().get('streamlit_proc')
if p is not None and p.poll() is None:
    p.terminate(); p.wait(timeout=10)
oldlog=globals().get('streamlit_log')
if oldlog is not None and not oldlog.closed: oldlog.close()
ngrok.kill()
streamlit_log=LOG_PATH.open('w',encoding='utf-8')
streamlit_proc=subprocess.Popen(['uv','run','streamlit','run','webui/Main.py',f'--server.port={PORT}','--server.address=0.0.0.0','--browser.gatherUsageStats=False','--client.toolbarMode=minimal','--server.showEmailPrompt=False'],cwd=REPO_DIR,stdout=streamlit_log,stderr=subprocess.STDOUT,text=True)
deadline=time.time()+90; server_ready=False
while time.time()<deadline:
    if streamlit_proc.poll() is not None: break
    try:
        with urlopen(f'http://127.0.0.1:{PORT}/_stcore/health',timeout=2) as r: server_ready=(r.status==200)
    except (URLError,TimeoutError): pass
    if server_ready: break
    time.sleep(2)
if not server_ready:
    streamlit_log.flush(); raise RuntimeError('Streamlit failed to start.\n'+LOG_PATH.read_text(encoding='utf-8',errors='replace')[-4000:])
public_tunnel=ngrok.connect(addr=f'http://127.0.0.1:{PORT}',proto='http',bind_tls=True)
print('MoneyPrinterTurbo is ready:'); print(public_tunnel.public_url); print(f'Server log: {LOG_PATH}')
